# Filtering NO2 Tempo parquet files including cloud data + all hours in output

In [2]:
import numpy as np 
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt 
import seaborn as sns
from datetime import datetime, timezone
from pathlib import Path
from tqdm import tqdm


In [7]:
PARQUET_BASE = Path('/global/scratch/users/rubypercival/TEMPO24/parquet_V04')
FILTERED_DIR = PARQUET_BASE / 'filtered'
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2024

In [8]:
mar_file = PARQUET_BASE / f'all_tempo_hours_no2_L3_V04_{YEAR}_03_folder'
apr_file = PARQUET_BASE / f'all_tempo_hours_no2_L3_V04_{YEAR}_04_folder'
may_file = PARQUET_BASE / f'all_tempo_hours_no2_L3_V04_{YEAR}_05_folder'
june_file = PARQUET_BASE / f'all_tempo_hours_no2_L3_V04_{YEAR}_06_folder'
july_file = PARQUET_BASE / f'all_tempo_hours_no2_L3_V04_{YEAR}_07_folder'

In [9]:
def filter_no2_dataframe(ddf, drop_extra_cols=False):
    #combine date and time into datetime
    ddf['datetime'] = dd.to_datetime(
        ddf['date'].astype(str) + ' ' + ddf['time'].astype(str),
        format='%Y-%m-%d %H:%M:%S.%f', errors='coerce')

    ddf['hour'] = ddf['datetime'].dt.floor('h').dt.time

    # quality filtering
    ddf_filt = ddf[
        (ddf['eff_cloud_fraction'] <= 0.10) &
        (ddf['snow_ice_fraction'] < 0.01) &
        (ddf['solar_zenith_angle'] < 70) &
        (ddf['no2'] > 0)]

    if drop_extra_cols:
        cols_to_drop = ['amf_cloud_fraction','snow_ice_fraction',
                        'eff_cloud_fraction','no2_uncertainty']
        ddf_filt = ddf_filt.drop(columns=cols_to_drop, errors='ignore')

    return ddf_filt

## first filter for march

In [10]:
mar_df = dd.read_parquet(mar_file, engine='pyarrow')


In [11]:
mar_df_filt = filter_no2_dataframe(mar_df, drop_extra_cols=False)


In [12]:
output_file = FILTERED_DIR / 'lowCF_filtered_mar_no2_V04.parquet'
mar_df_filt.to_parquet(output_file, engine='pyarrow', write_index=False)


/global/software/rocky-8.x86_64/manual/modules/langs/anaconda3/2024.02-1/lib/python3.11/site-packages/dask/dataframe/_pyarrow.py:82: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df = df.astype(dtypes, copy=False)


## first filtering for april

In [13]:
apr_df = dd.read_parquet(apr_file, engine='pyarrow')


In [14]:
apr_df_filt = filter_no2_dataframe(apr_df, drop_extra_cols=False)


In [15]:
output_file = FILTERED_DIR / 'lowCF_filtered_apr_no2_V04.parquet'
apr_df_filt.to_parquet(output_file, engine='pyarrow', write_index=False)


## now filtering for may

In [16]:
may_df = dd.read_parquet(may_file, engine='pyarrow')


In [17]:
may_df_filt = filter_no2_dataframe(may_df, drop_extra_cols=False)


In [18]:
output_file = FILTERED_DIR / 'lowCF_filtered_may_no2_V04.parquet'
may_df_filt.to_parquet(output_file, engine='pyarrow', write_index=False)


## now filtering for june

In [19]:
june_df = dd.read_parquet(june_file, engine='pyarrow')


In [20]:
june_df_filt = filter_no2_dataframe(june_df, drop_extra_cols=False)


In [21]:
output_file = FILTERED_DIR / 'lowCF_filtered_june_no2_V04.parquet'
june_df_filt.to_parquet(output_file, engine='pyarrow', write_index=False)


## filter for july 

In [22]:
july_df = dd.read_parquet(july_file, engine='pyarrow')


In [23]:
july_df_filt = filter_no2_dataframe(july_df, drop_extra_cols=False)


In [24]:
output_file = FILTERED_DIR / 'lowCF_filtered_july_no2_V04.parquet'
july_df_filt.to_parquet(output_file, engine='pyarrow', write_index=False)

## concatenate all no2 data to one df 

In [26]:
mar_file = FILTERED_DIR / 'lowCF_filtered_mar_no2_V04.parquet'
apr_file = FILTERED_DIR / 'lowCF_filtered_apr_no2_V04.parquet'
may_file = FILTERED_DIR / 'lowCF_filtered_may_no2_V04.parquet'
june_file = FILTERED_DIR / 'lowCF_filtered_june_no2_V04.parquet'
july_file = FILTERED_DIR / 'lowCF_filtered_july_no2_V04.parquet'


In [27]:
mar_df = dd.read_parquet(mar_file, engine='pyarrow')
apr_df = dd.read_parquet(apr_file, engine='pyarrow')
may_df = dd.read_parquet(may_file, engine='pyarrow')
june_df = dd.read_parquet(june_file, engine='pyarrow')
july_df = dd.read_parquet(july_file, engine='pyarrow')


In [28]:
all_no2 = dd.concat([mar_df, apr_df, may_df, june_df, july_df])


In [30]:
all_no2 = dd.concat([mar_df, apr_df, may_df, june_df, july_df])

output_file = FILTERED_DIR / 'lowCF_filtered_mar-july_no2_V04.parquet'
all_no2.to_parquet(output_file, engine='pyarrow', write_index=False)
print('Saved to:', output_file)

Saved to: /global/scratch/users/rubypercival/TEMPO24/parquet_V04/filtered/lowCF_filtered_mar-july_no2_V04.parquet


In [31]:
output_file = FILTERED_DIR / 'lowCF_filtered_mar-july_no2_V04.parquet'

all_no2 = pd.read_parquet(
    output_file,
    columns=['amf_cloud_fraction', 'eff_cloud_fraction', 'snow_ice_fraction'],
    engine='pyarrow')


In [32]:
all_no2

,amf_cloud_fraction,eff_cloud_fraction,snow_ice_fraction
0,0.153971,0.043989,0.0
1,0.157731,0.045107,0.0
2,0.156692,0.044594,0.0
3,0.187704,0.054759,0.0
4,0.188946,0.055206,0.0
...,...,...,...
509667287,0.144979,0.038028,0.0
509667288,0.111259,0.028382,0.0
509667289,0.096756,0.024263,0.0
509667290,0.096663,0.024227,0.0


In [33]:
all_no2['amf_cloud_fraction'].describe()

count    5.096673e+08
mean     1.080249e-01
std      9.925995e-02
min      0.000000e+00
25%      2.190562e-02
50%      7.975608e-02
75%      1.772205e-01
max      5.876711e-01
Name: amf_cloud_fraction, dtype: float64

In [34]:
all_no2['eff_cloud_fraction'].describe()

count    5.096673e+08
mean     2.818813e-02
std      2.753770e-02
min      0.000000e+00
25%      5.099901e-03
50%      1.906426e-02
75%      4.533011e-02
max      1.000000e-01
Name: eff_cloud_fraction, dtype: float64

In [35]:
all_no2['snow_ice_fraction'].describe()

count    5.096673e+08
mean     1.561745e-06
std      9.726801e-05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.999962e-03
Name: snow_ice_fraction, dtype: float64